<a href="https://colab.research.google.com/github/shardeep-x/agentic-ai-and-gen-ai/blob/foundation_of_gen_ai_llm/%5BC2%5D%5BModule2%5D_Introduction_to_prompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to prompting

## Contents
1. Prompting an LLM using transformers
2. zero-shot prompting
3. Few-shot prompting
4. Chain-of-thoughts prompting
5. Appendix

## Prompting an LLM using transformers

Using the transformers library, we can download the weights of a model and run the LLM on local resources.

**Step 0**: Load the required libraries (`torch`, `transformers`)

**Step 1**: Find the model name from https://huggingface.co/models

Examples:
- **Llama 3.2**: meta-llama/Llama-3.2-3B-Instruct
- **Qwen 2.5**: Qwen/Qwen2.5-3B
- **Phi 4**: microsoft/Phi-4-mini-instruct
- **Gemma 4**: google/gemma-4-E4B-it
- **GLM 4.6**: zai-org/GLM-4.6V-Flash
- **Bonsai**: prism-ml/Ternary-Bonsai-4B-mlx-2bit

**Step 2**: Load model and tokenizer

`AutoTokenizer.from_pretrained()`
`AutoModelForCausalLM.from_pretrained()`

**Step 3**: Create the prompt template

**Step 4**: Tokenize the prompt `tokenizer.apply_chat_template()`

**Step 5**: Prompt the model using `model.generate()`

**Step 6**: Parse the generate tokens using `tokenizer.decode()`

>Note on `bfloat16`:
>
>float16: exponent (5 bits), mantissa (10 bits)\
>bfloat16: exponent (8 bits), mantissa (7 bits)



In [ ]:
## CODE HERE ##
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

name = "unsloth/Llama-3.2-3B-Instruct"

tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(
    name, torch_dtype=torch.bfloat16, device_map="cuda"
)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

In [ ]:
messages = [{"role" : "system", "content" : "You are an helpful assitant"},
            {"role" : "user", "content" : "What is the captial fo france"}]

ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
print(ids)


{'input_ids': tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,    914,  10263,    220,   2366,     21,    271,   2675,    527,
            459,  11190,   1089,  52044, 128009, 128006,    882, 128007,    271,
           3923,    374,    279,   6563,    532,  12018,  48687, 128009, 128006,
          78191, 128007,    271]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}


In [ ]:
out = model.generate(**ids, max_new_tokens=500, temperature=0.7)

print(out)

[transformers] Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,    914,  10263,    220,   2366,     21,    271,   2675,    527,
            459,  11190,   1089,  52044, 128009, 128006,    882, 128007,    271,
           3923,    374,    279,   6563,    532,  12018,  48687, 128009, 128006,
          78191, 128007,    271,    791,   6864,    315,   9822,    374,  12366,
             13, 128009]], device='cuda:0')


In [ ]:
decoded = tok.decode(out[0, ids["input_ids"].shape[-1]:], skip_special_tokens=True, )

print(decoded)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The capital of France is Paris.


**Takeaway:**
1. How to load an open-source LLM into memory
2. How to tokenize prompts and use it to prompt a model
3. How the model returns its responses

### Using pipeline

Pipelines are a wrapper around the tokenizer and model that combines all the steps into a single function call. There are various tasks that a pipeline can do including `text-classification`, `text-generation`, and `token-classification`.

In [ ]:
## CODE HERE ##
from transformers import pipeline

model_pipeline = pipeline(
    "text-generation",
    model=name,
    max_new_tokens=500,
    temperature=0.7,
    top_k=10,
    top_p=0.8,
)


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'top_p', 'top_k', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
output = model_pipeline(messages)
print(output)

[transformers] Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'system', 'content': 'You are an helpful assitant'}, {'role': 'user', 'content': 'What is the captial fo france'}, {'role': 'assistant', 'content': 'The capital of France is Paris.'}]}]


In [ ]:
print(output[0]["generated_text"][-1]["content"])

The capital of France is Paris.


**Takeaway:**
1. How to simplify tokenization and prompting into a single function call.
2. How to specify the parameters like temperature and top_k.
3. How the model returns its responses

### Utility functions

`pipeline` returns the responses in a list of dictionary format, simlar to `tokenizer.decode`. Let's declare some functions to do the repetitive tasks.

#### 1. Format the prompt as a chat template

In [ ]:
def make_message(system_content, user_content):
  return [
      {"role": "system", "content": system_content},
      {"role": "user", "content": user_content},
  ]

#### 2. Parse the response out of the returned format

In [ ]:
def fetch_response(response):
  return response[0]["generated_text"][-1]["content"]

**Takeaway:**
Instead of formatting the data everytime, we declare utility functions and use them as and when required.

## Zero-shot prompting

Scenario:

Standardizing unstructured release notes into a structured JSON payload for an automated deployment pipeline.

System prompt
> You are an automated DevOps data-ingestion microservice. Your job is to convert raw developer text into clean, strict JSON payloads matching the requested schema. Never add conversational filler, markdown formatting blocks outside the JSON, or extra text.

User prompt
> Convert the following unstructured release notes into a valid JSON object.
The JSON must contain three keys: "version", "breaking_changes" (array of strings), and "services_impacted" (array of strings).
>
> Release Notes:
"We just rolled out v2.14.0! Big update to the payments engine—we deprecated the old v1/charge endpoint, so any integration still hitting that will fail. Also updated the notification microservice to support webhooks, and patched a minor memory leak in the billing-sync worker."

In [ ]:
## CODE HERE ##
system_prompt = "You are an automated DevOps data-ingestion microservice. Your job is to convert raw developer text into clean, strict JSON payloads matching the requested schema. Never add conversational filler, markdown formatting blocks outside the JSON, or extra text."
user_prompt = """Convert the following unstructured release notes into a valid JSON object.
The JSON must contain three keys: "version", "breaking_changes" (array of strings), and "services_impacted" (array of strings).
Release Notes:
We just rolled out v2.14.0! Big update to the payments engine—we deprecated the old v1/charge endpoint, so any integration still hitting that will fail. Also updated the notification microservice to support webhooks, and patched a minor memory leak in the billing-sync worker."""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))



[transformers] Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{"version": "2.14.0", "breaking_changes": ["deprecated the old v1/charge endpoint"], "services_impacted": ["payments engine", "notification microservice"]}


## Few-shot prompting

Scenario: Solving a classification problem by providing explicit input-output pairs that provides the taxonomy.

System prompt:
> You are an enterprise incident response parser. Map incoming customer tickets strictly to the allowed tags and metadata structure demonstrated in the examples. Do not invent new tags.

User prompt:
> Classify incoming incident reports into our internal service tags using only these valid tags:
> - IAM_TOKEN_ROTATION_DESYNC
> - DB_CONNECTION_POOL_EXHAUSTED
> - GATEWAY_PAYLOAD_TOO_LARGE
>
> Examples:
>
> Input: "Cron job failed with 401 unauthorized immediately after secrets manager rotated the DB password."
>
> Output: {"tag": "IAM_TOKEN_ROTATION_DESYNC", "severity": "P2"}
>
> Input: "API latency spiked to 12s. Postgres logs show 'fatal: remaining connection slots are reserved'."
>
> Output: {"tag": "DB_CONNECTION_POOL_EXHAUSTED", "severity": "P1"}
>
> Input: "Post request to /analytics/bulk-import returned 413. Payload size was 45MB."
>
> Output: {"tag": "GATEWAY_PAYLOAD_TOO_LARGE", "severity": "P3"}
>
> Input: "Our automated batch jobs failed at midnight. The dashboard says 'Error 401: Invalid Token Signature' when calling the data warehouse service, but our API key was renewed yesterday."
>
> Output:

In [ ]:
## CODE HERE ##

system_prompt = "You are an enterprise incident response parser. Map incoming customer tickets strictly to the allowed tags and metadata structure demonstrated in the examples. Do not invent new tags."

user_prompt = """Classify incoming incident reports into our internal service tags using only these valid tags:

IAM_TOKEN_ROTATION_DESYNC
DB_CONNECTION_POOL_EXHAUSTED
GATEWAY_PAYLOAD_TOO_LARGE
Examples:

Input: "Cron job failed with 401 unauthorized immediately after secrets manager rotated the DB password."

Output: {"tag": "IAM_TOKEN_ROTATION_DESYNC", "severity": "P2"}

Input: "API latency spiked to 12s. Postgres logs show 'fatal: remaining connection slots are reserved'."

Output: {"tag": "DB_CONNECTION_POOL_EXHAUSTED", "severity": "P1"}

Input: "Post request to /analytics/bulk-import returned 413. Payload size was 45MB."

Output: {"tag": "GATEWAY_PAYLOAD_TOO_LARGE", "severity": "P3"}

Input: "Our automated batch jobs failed at midnight. The dashboard says 'Error 401: Invalid Token Signature' when calling the data warehouse service, but our API key was renewed yesterday."""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

[transformers] Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Here's a Python class that can parse incoming incident reports and classify them into internal service tags:

```python
class IncidentReporter:
    def __init__(self):
        self.tags = {
            "Cron job failed with 401 unauthorized immediately after secrets manager rotated the DB password": {
                "tag": "IAM_TOKEN_ROTATION_DESYNC",
                "severity": "P2"
            },
            "API latency spiked to 12s. Postgres logs show 'fatal: remaining connection slots are reserved'": {
                "tag": "DB_CONNECTION_POOL_EXHAUSTED",
                "severity": "P1"
            },
            "Post request to /analytics/bulk-import returned 413. Payload size was 45MB": {
                "tag": "GATEWAY_PAYLOAD_TOO_LARGE",
                "severity": "P3"
            },
            "Our automated batch jobs failed at midnight. The dashboard says 'Error 401: Invalid Token Signature' when calling the data warehouse service, but our API key was renewed yesterd

## Chain-of-thougths prompting

Scenario: Resolving the complex business logic by using system instructions and user prompt structures that mandate step-by-step intermediate calculation.

System prompt:
> You are an enterprise compliance auditor. Whenever evaluating complex multi-step SLA calculations, you must always show your step-by-step arithmetic and logic checks before emitting the final percentage to ensure mathematical accuracy.

User prompt:
>Determine the SLA credit percentage for Enterprise customers based on downtime incidents.
>
>Rules:
>- Tier: Enterprise
>- SLA Target: 99.9% monthly uptime.
>- Credit structure:
>  * 0 to 43 mins downtime: 0% credit
>  * 44-120 mins downtime: 10% credit
>  * 121-300 mins downtime: 25% credit
>  * \>300 mins downtime: 50% credit
>  * If incident occurred during a scheduled maintenance window, subtract maintenance mins from total outage before evaluating credit.
>
>Input:
>Customer Tier: Enterprise
>Total Outage Duration: 340 minutes
>Scheduled Maintenance Window: 50 minutes
>
>Instructions:
>Think through this step-by-step:
>1. Identify total outage and maintenance window.
>2. Calculate net billable downtime (Total Outage minus Scheduled Maintenance).
>3. Compare net billable downtime against the credit structure tiers.
>4. State the final refund credit percentage.

In [ ]:
## CODE HERE ##

system_prompt = "You are an enterprise compliance auditor. Whenever evaluating complex multi-step SLA calculations, you must always show your step-by-step arithmetic and logic checks before emitting the final percentage to ensure mathematical accuracy."

user_prompt = """Determine the SLA credit percentage for Enterprise customers based on downtime incidents.

Rules:
- Tier: Enterprise
- SLA Target: 99.9% monthly uptime.
- Credit structure:
  * 0 to 43 mins downtime: 0% credit
  * 44-120 mins downtime: 10% credit
  * 121-300 mins downtime: 25% credit
  * >300 mins downtime: 50% credit
  * If incident occurred during a scheduled maintenance window, subtract maintenance mins from total outage before evaluating credit.

Input:
Customer Tier: Enterprise
Total Outage Duration: 340 minutes Scheduled Maintenance Window: 50 minutes

Instructions: Think through this step-by-step:

Identify total outage and maintenance window.
Calculate net billable downtime (Total Outage minus Scheduled Maintenance).
Compare net billable downtime against the credit structure tiers.
State the final refund credit percentage. """

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

[transformers] Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


To determine the SLA credit percentage, we'll follow the step-by-step process.

**Step 1: Identify total outage and maintenance window**

Given input:
- Customer Tier: Enterprise
- Total Outage Duration: 340 minutes
- Scheduled Maintenance Window: 50 minutes

**Step 2: Calculate net billable downtime**

To find the net billable downtime, we subtract the scheduled maintenance window from the total outage duration.

Net Billable Downtime = Total Outage Duration - Scheduled Maintenance Window
Net Billable Downtime = 340 minutes - 50 minutes
Net Billable Downtime = 290 minutes

**Step 3: Compare net billable downtime against the credit structure tiers**

We'll compare the net billable downtime to the credit structure tiers.

- Tier 1: 0-43 minutes downtime, 0% credit
- Tier 2: 44-120 minutes downtime, 10% credit
- Tier 3: 121-300 minutes downtime, 25% credit
- Tier 4: >300 minutes downtime, 50% credit

Since the net billable downtime (290 minutes) falls within the range of Tier 3 (121-300 

## Appendix

### Prompting a hosted model

1. HTTP requests -  No libraries are required. Send queries to an LLM hosted on a web server using an API endpoint URL and an API key from the LLM provider.
2. Official libraries - Python libraries provided by the organization. Sends a request to the server and fetches the response. Requires an API key.

#### HTTP requests

The prompt is sent to an API endpoint and the response is returned as a json. No libraries are required to be installed.

In [ ]:
import os, requests
from dotenv import load_dotenv
load_dotenv()

PROMPT = "What is the capital of France?"

##### Anthropic

In [ ]:
r = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers={
        "Content-Type": "application/json",
        "X-API-Key": os.environ["ANTHROPIC_API_KEY"],
        "anthropic-version": "2023-06-01",
    },
    json={
        "model": "claude-sonnet-4-5",
        "max_tokens": 256,
        "messages": [{"role": "user", "content": PROMPT}],
    },
)
print(r.json()["content"][0]["text"])

KeyError: 'ANTHROPIC_API_KEY'

##### OpenAI

In [ ]:
r = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
    json={
        "model": "gpt-4o-mini",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": PROMPT},
        ],
    },
)
print(r.json()["choices"][0]["message"]["content"])

##### Gemini

In [ ]:
r = requests.post(
    "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent",
    headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]},
    json={
        "systemInstruction": {"parts": [{"text": "You are a helpful assistant."}]},
        "contents": [{"role": "user", "parts": [{"text": PROMPT}]}],
        "generationConfig": {"maxOutputTokens": 256},
    },
)
print(r.json()["candidates"][0]["content"]["parts"][0]["text"])

##### Qwen

In [ ]:
r = requests.post(
    "https://dashscope-intl.aliyuncs.com/compatible-mode/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.environ['DASHSCOPE_API_KEY']}"},
    json={"model": "qwen-plus", "messages": [{"role": "user", "content": PROMPT}]},
)
print(r.json()["choices"][0]["message"]["content"])

#### Official libraries

##### Anthropic

In [ ]:
from anthropic import Anthropic

client = Anthropic()

msg = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=256,
    system="You are a helpful assistant.",
    messages=[{"role": "user", "content": PROMPT}],
)
print(msg.content[0].text)

##### OpenAI

In [ ]:
from openai import OpenAI

client = OpenAI()

r = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": PROMPT}],
)
print(r.choices[0].message.content)

##### Gemini

In [ ]:
from google import genai
from google.genai import types

client = genai.Client()          # reads GEMINI_API_KEY

r = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=PROMPT,
    config=types.GenerateContentConfig(system_instruction="You are a terse materials scientist."),
)
print(r.text)

##### Qwen

In [ ]:
client = OpenAI(base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
                api_key=os.environ["DASHSCOPE_API_KEY"])
# local vLLM on loki:  base_url="http://localhost:8000/v1", api_key="EMPTY"
r = client.chat.completions.create(model="qwen-plus",
                                   messages=[{"role": "user", "content": PROMPT}])
print(r.choices[0].message.content)

### Codes

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(
    name, torch_dtype=torch.bfloat16, device_map="cuda"
)

messages = [{"role": "user", "content": "What is the capital of France?"}]
ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")

out = model.generate(ids, max_new_tokens=128, do_sample=True, temperature=0.7)
print(tok.decode(out[0, ids.shape[-1]:], skip_special_tokens=True))

In [ ]:
from transformers import pipeline

model_pipeline = pipeline(
    "text-generation",
    model=name,
    max_new_tokens=500,
    temperature=0.7,
    top_k=10,
    top_p=0.85,
)

print(model_pipeline(messages))

### Zero-shot prompting examples

In [ ]:
system_prompt = "You are a helpful math expert."

user_prompt = """
What is the result of: 128 + 256
Answer:
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

In [ ]:
# Generate a property listing from a fact sheet
fact_sheet_house = """
OVERVIEW
- 4-bedroom, 3-bathroom single-family home in a quiet suburban neighbourhood.
- Built in 2015, renovated in 2022.

FEATURES
- Modern kitchen with granite countertops and stainless-steel appliances.
- Hardwood flooring in living areas; carpet in bedrooms.
- Master suite with walk-in closet and soaking tub.

LOCATION
- Sector 3, Saltlake, Kolkata, India
- Near top-rated schools, parks, and public transport.

DETAILS
- 2,450 sq ft interior | 6,000 sq ft lot
- HOA: Rs.50/month | Property tax: ~Rs.6,200/year
"""

system_prompt = "You are a real estate marketing assistant who writes compelling property descriptions."
user_prompt = "Create a property description based on the following fact sheet: ```" + fact_sheet_house + "```"

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

In [ ]:
# Sentiment classification from reviews
headphones_review = """
I've been using these headphones for about two weeks now.
The sound quality is excellent: clear highs and strong bass.
They're super comfortable even after hours of use.
Battery life lasts me through multiple workdays.
Only downside is the case feels a bit cheap, but overall I'm very happy.
"""

system_prompt = "You are a sentiment analysis assistant. Determine whether a product review is positive or negative."

user_prompt = (
    "What is the sentiment of the following review (delimited by triple backticks)?\n"
    "Give your answer as a single word: positive or negative.\n\n"
    "Review: ```" + headphones_review + "```"
)

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

Scenario: Categorizing customer support tickets using strict internal engineering microservice tags without providing the list of tags.

System prompt
> You are an automated IT service desk ticket router. Your job is to read incoming support tickets and categorize them into appropriate incident response tags.

User prompt
> Classify the following customer ticket into one of our internal incident response tags.
>
> Ticket: "Our automated batch jobs failed at midnight. The dashboard says 'Error 401: Invalid Token Signature' when calling the data warehouse service, but our API key was renewed yesterday."

### Few-shot prompting examples

In [ ]:
# Text translation with a consistent format
# Useful when you need the model to follow a strict input/output pattern.

system_prompt = "You are a translation assistant. Translate English sentences to French."

user_prompt = """Here are some examples:

English: Good morning, how are you?
French: Bonjour, comment allez-vous?

English: The weather is beautiful today.
French: Le temps est magnifique aujourd'hui.

English: I would like a cup of coffee, please.
French: Je voudrais une tasse de cafe, s'il vous plait.

Now translate:
English: Where is the nearest hospital?
French:
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

In [ ]:
# Movie review classification
system_prompt = "You are a movie review classifier. Classify reviews as POSITIVE or NEGATIVE."

user_prompt = """Here are some examples:

Review: "An absolute masterpiece. The acting was superb and the story kept me hooked."
Sentiment: POSITIVE

Review: "Terrible plot, wooden acting, and a predictable ending. Complete waste of time."
Sentiment: NEGATIVE

Review: "A beautiful film with stunning visuals and a touching story."
Sentiment: POSITIVE

Now classify this review:
Review: "The movie was boring and too long. I almost fell asleep in the second half."
Sentiment:
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

In [ ]:
# Parsing data into predefined format
system_prompt = "You are a data extraction assistant. Extract information from customer feedback into JSON."

user_prompt = """Here are some examples:

Feedback: "The delivery was super fast, arrived the next day! But the packaging was damaged."
Output: {"delivery": "fast", "packaging": "damaged", "overall_sentiment": "mixed"}

Feedback: "Product quality is amazing and the price is very reasonable."
Output: {"product_quality": "amazing", "price": "reasonable", "overall_sentiment": "positive"}

Now extract information from this feedback:
Feedback: "Customer support was very helpful, but the product stopped working after a week."
Output:
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

Scenario: Calculating customer SLA breach refunds using multi-tiered business rules where few-shot pattern matching falls short on arithmetic logic.

System prompt:
> You are a customer success billing bot. Compute SLA refund credit percentages for enterprise clients based on downtime policy rules.

User prompt:
> Determine the SLA credit percentage for Enterprise customers based on downtime incidents.
>
> Rules:
> - Tier: Enterprise
> - SLA Target: 99.9% monthly uptime (~43 minutes allowed downtime).
> - Credit structure:
>   * 44-120 mins downtime: 10% credit
>   * 121-300 mins downtime: 25% credit
>   * \>300 mins downtime: 50% credit
>   * If incident occurred during a scheduled maintenance window, subtract maintenance mins before calculating.
>
> Examples:
> Input: Tier: Enterprise | Total Outage: 180 mins | Scheduled Maint: 0 mins\
Output: Credit: 25%
>
>Input: Tier: Enterprise | Total Outage: 200 mins | Scheduled Maint: 100 mins\
Output: Credit: 10%
>
>Input: Tier: Enterprise | Total Outage: 450 mins | Scheduled Maint: 60 mins\
Output: Credit: 25%
>
>Input: Tier: Enterprise | Total Outage: 340 mins | Scheduled Maint: 50 mins\
>Output:

### Chain-of-thoughts prompting examples

In [ ]:
# Zero-shot CoT
# The magic phrase "Let's think step by step" triggers step-by-step reasoning.

system_prompt = "You are a helpful math tutor."

user_prompt = """A store sells apples for Rs.15 each and oranges for Rs.20 each.
If Maya buys 4 apples and 3 oranges, how much does she spend in total?

Let's think step by step.
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

In [ ]:
# Few-shot CoT
# Examples show the model *how* to reason, not just the final answer.

system_prompt = "You are a logical reasoning assistant. Always show your reasoning before the final answer."

user_prompt = """Here are some examples:

Question: A train travels at 60 km/h. How far does it go in 2.5 hours?
Reasoning:
  Step 1: Distance = Speed x Time
  Step 2: Distance = 60 x 2.5 = 150 km
Answer: 150 km

Question: A shop has 200 items. 40% are sold. How many remain?
Reasoning:
  Step 1: Items sold = 40% of 200 = 0.40 x 200 = 80
  Step 2: Items remaining = 200 - 80 = 120
Answer: 120 items

Now solve:
Question: A recipe needs 3 cups of flour for 12 cookies. How many cups are needed for 40 cookies?
Reasoning:
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

In [ ]:
# CoT for logical / commonsense reasoning

system_prompt = "You are a logical reasoning assistant. Think step by step before answering."

user_prompt = """
All mammals are warm-blooded.
Whales are mammals.
Dolphins are mammals.
Sharks are fish, not mammals.

Question: Are whales and dolphins warm-blooded? Is a shark warm-blooded?
Let's think step by step.
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))